# Demo 5: Adatminőség alapok

**Kapcsolódó diák:** 38–45 (DQ dimenziók, profiling, DQ score, pipeline gate)

**Előfeltétel:** `docker compose up -d` – JupyterLab: http://localhost:8888 (token: demo)

Tartalom:
1. Adatprofiling – NULL arány, kardinalitás, eloszlás
2. DQ metrikák – mind a 6 dimenzió mérése
3. Súlyozott DQ score számítás
4. DQ pipeline gate – PASS / WARN / FAIL döntés


In [1]:
!pip install -q pandas numpy

## Minta adatset

Szándékosan hibás adatot használunk: duplikált `order_id`, NULL `customer_id`,
negatív `amount`, jövőbeli dátum, érvénytelen email.


In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'order_id':    [1, 2, 2, 4, 5, 6, 7, 8, 9, 10],       # duplikált: 2
    'customer_id': [101, 102, 102, None, 105, 106, None, 108, 109, 110],
    'amount':      [12500, 3200, 3200, 8750, -100, 99999, 450, 0, 15000, 7300],
    'order_date':  ['2024-01-15', '2024-01-16', '2024-01-16', '2024-01-17',
                    '2025-12-31',  # jövőbeli!
                    '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22'],
    'status':      ['shipped', 'pending', 'pending', 'shipped', 'cancelled',
                    'SHIPPED',   # inkonzisztens
                    'pending', 'shipped', 'shipped', 'cancelled'],
    'email':       ['a@b.hu', None, None, 'c@d.hu', 'nem_email',  # érvénytelen
                    'e@f.hu', 'g@h.hu', None, 'i@j.hu', 'k@l.hu'],
})

print(f'Adatset: {len(df)} sor, {len(df.columns)} oszlop')
print(df.to_string(index=False))

Adatset: 10 sor, 6 oszlop
 order_id  customer_id  amount order_date    status     email
        1        101.0   12500 2024-01-15   shipped    a@b.hu
        2        102.0    3200 2024-01-16   pending      None
        2        102.0    3200 2024-01-16   pending      None
        4          NaN    8750 2024-01-17   shipped    c@d.hu
        5        105.0    -100 2025-12-31 cancelled nem_email
        6        106.0   99999 2024-01-18   SHIPPED    e@f.hu
        7          NaN     450 2024-01-19   pending    g@h.hu
        8        108.0       0 2024-01-20   shipped      None
        9        109.0   15000 2024-01-21   shipped    i@j.hu
       10        110.0    7300 2024-01-22 cancelled    k@l.hu


## 1. Adatprofiling


In [3]:
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        null_count  = df[col].isna().sum()
        unique_vals = df[col].nunique(dropna=True)
        row = {
            'oszlop':   col,
            'dtype':    str(df[col].dtype),
            'null_db':  null_count,
            'null_%':   round(null_count / len(df) * 100, 1),
            'egyedi':   unique_vals,
        }
        if pd.api.types.is_numeric_dtype(df[col]):
            row['min']  = df[col].min()
            row['max']  = df[col].max()
            row['mean'] = round(df[col].mean(), 1)
        rows.append(row)
    return pd.DataFrame(rows)

profile = profile_dataframe(df)
print('=== Adatprofil ===')
print(profile[['oszlop', 'dtype', 'null_%', 'egyedi']].to_string(index=False))

=== Adatprofil ===
     oszlop   dtype  null_%  egyedi
   order_id   int64     0.0       9
customer_id float64    20.0       7
     amount   int64     0.0       9
 order_date  object     0.0       9
     status  object     0.0       4
      email  object    30.0       7


## 2. DQ metrikák

In [4]:
import re
from datetime import date

class DQMetrics:
    def __init__(self, df): self.df = df; self.n = len(df)

    def completeness(self, col): return 1 - self.df[col].isna().mean()

    def validity_amount_positive(self): return (self.df['amount'] > 0).sum() / self.n

    def validity_email_format(self, col='email'):
        pat = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
        non_null = self.df[col].dropna()
        valid = non_null.apply(lambda v: bool(re.match(pat, str(v)))).sum()
        return valid / self.n

    def validity_date_not_future(self, col='order_date'):
        today  = pd.Timestamp(date.today())
        parsed = pd.to_datetime(self.df[col], errors='coerce')
        return (parsed <= today).sum() / self.n

    def uniqueness(self, col):
        return 1 - self.df[col].duplicated(keep=False).sum() / self.n

    def consistency_status(self, col='status', allowed=None):
        if allowed is None: allowed = {'shipped', 'pending', 'cancelled'}
        return self.df[col].str.lower().isin(allowed).sum() / self.n

    def timeliness_simulated(self):
        rng = np.random.default_rng(42)
        return (rng.uniform(0, 48, self.n) <= 24).mean()


m = DQMetrics(df)
results = {
    'completeness_customer_id': m.completeness('customer_id'),
    'completeness_email':       m.completeness('email'),
    'validity_amount_positive': m.validity_amount_positive(),
    'validity_email_format':    m.validity_email_format(),
    'validity_date_not_future': m.validity_date_not_future(),
    'uniqueness_order_id':      m.uniqueness('order_id'),
    'consistency_status':       m.consistency_status(),
    'timeliness_24h':           m.timeliness_simulated(),
}

print('=== DQ Metrikák ===')
for k, v in results.items():
    bar = '█' * int(v * 20) + '░' * (20 - int(v * 20))
    icon = '✅' if v >= 0.9 else ('⚠️' if v >= 0.75 else '❌')
    print(f'{icon} {k:35s} {bar} {v*100:5.1f}%')

=== DQ Metrikák ===
⚠️ completeness_customer_id            ████████████████░░░░  80.0%
❌ completeness_email                  ██████████████░░░░░░  70.0%
⚠️ validity_amount_positive            ████████████████░░░░  80.0%
❌ validity_email_format               ████████████░░░░░░░░  60.0%
✅ validity_date_not_future            ████████████████████ 100.0%
⚠️ uniqueness_order_id                 ████████████████░░░░  80.0%
✅ consistency_status                  ████████████████████ 100.0%
❌ timeliness_24h                      ████████░░░░░░░░░░░░  40.0%


## 3. Súlyozott DQ score


In [5]:
def compute_dq_score(metric_results: dict, weights: dict = None) -> float:
    if weights is None:
        n = len(metric_results)
        weights = {k: 1 / n for k in metric_results}
    return round(sum(metric_results[k] * weights.get(k, 0) for k in metric_results) * 100, 2)


weights = {
    'completeness_customer_id': 0.20,
    'completeness_email':       0.10,
    'validity_amount_positive': 0.20,
    'validity_email_format':    0.10,
    'validity_date_not_future': 0.15,
    'uniqueness_order_id':      0.15,
    'consistency_status':       0.05,
    'timeliness_24h':           0.05,
}

score = compute_dq_score(results, weights)
print(f'=== DQ Score: {score}/100 ===')
if score >= 90:   print('🟢 Elfogadható – Gold rétegbe engedhető')
elif score >= 75: print('🟡 Figyelem – javítás ajánlott, Silver szinten tartandó')
else:             print('🔴 Nem megfelelő – pipeline megállítva, riasztás küldve')

=== DQ Score: 79.0/100 ===
🟡 Figyelem – javítás ajánlott, Silver szinten tartandó


## 4. DQ pipeline gate

A DQ gate a Bronze→Silver és Silver→Gold átmeneteken dönt:
PASS = továbbenged, WARN = figyelmeztet, FAIL = pipeline megállítás.


In [6]:
class DQPipelineGate:
    def __init__(self, threshold_warn=75.0, threshold_fail=60.0):
        self.threshold_warn = threshold_warn
        self.threshold_fail = threshold_fail

    def run(self, df: pd.DataFrame, stage: str = 'silver') -> float:
        m = DQMetrics(df)
        res = {
            'completeness': m.completeness('customer_id'),
            'validity':     m.validity_amount_positive(),
            'uniqueness':   m.uniqueness('order_id'),
            'consistency':  m.consistency_status(),
        }
        score = compute_dq_score(res)
        print(f'\n=== DQ Gate [{stage.upper()}] – Score: {score}/100 ===')
        for k, v in res.items():
            print(f'  {k:15}: {v*100:.1f}%')
        if score >= 90:
            print(f'🟢 PASS – továbbenged a {stage} rétegbe')
        elif score >= self.threshold_warn:
            print(f'🟡 WARN – figyelmeztetés, adat továbbenged')
        elif score >= self.threshold_fail:
            print(f'🔴 FAIL – pipeline megáll!')
            raise RuntimeError(f'DQ gate FAIL: score {score} < {self.threshold_fail}')
        return score


gate = DQPipelineGate()
gate.run(df, stage='silver')


=== DQ Gate [SILVER] – Score: 85.0/100 ===
  completeness   : 80.0%
  validity       : 80.0%
  uniqueness     : 80.0%
  consistency    : 100.0%
🟡 WARN – figyelmeztetés, adat továbbenged


85.0